# Train your own signs for Gesture Glasses

**Before you start:** make a zip file called `signs.zip` with one folder per sign:

```
signs/
  none/        <- REQUIRED: photos of hands doing nothing special, or random hands
  hello/       <- REQUIRED: an open palm facing the glasses (the wave that starts reading)
  help/
  water/
  thank_you/
```

- 50-100 photos per folder. Different people, backgrounds and lighting.
- The whole hand must be visible. Take them from where the glasses wearer would stand (facing the signer).
- The folder name is what the glasses will show (`thank_you` shows as "thank you").
- Keep the `hello` folder: Gesture Glasses reads nothing until the signer waves (consent), and it
  recognizes the wave by this sign. Without it the app waits for a wave forever.
- Prefer signs held still: the model looks at one frame at a time, so moving signs are unreliable.

Then run each cell below, top to bottom (click it, press **Shift + Enter**).

In [ ]:
# 1. Install the training tool (takes about 2 minutes)
!pip install -q mediapipe-model-maker

In [ ]:
# 2. Upload signs.zip (a 'Choose files' button appears below)
from google.colab import files
files.upload()
!rm -rf data && mkdir data && unzip -q signs.zip -d data
!ls data/signs

In [ ]:
# 3. Train (5-15 minutes). Photos where no hand is found are skipped automatically.
from mediapipe_model_maker import gesture_recognizer

data = gesture_recognizer.Dataset.from_folder(
    dirname="data/signs",
    hparams=gesture_recognizer.HandDataPreprocessingParams(),
)
train_data, rest = data.split(0.8)
validation_data, test_data = rest.split(0.5)

options = gesture_recognizer.GestureRecognizerOptions(
    hparams=gesture_recognizer.HParams(export_dir="exported_model", epochs=20),
)
model = gesture_recognizer.GestureRecognizer.create(
    train_data=train_data, validation_data=validation_data, options=options,
)
loss, accuracy = model.evaluate(test_data, batch_size=1)
print(f"Accuracy on photos it never saw: {accuracy:.0%}")

In [ ]:
# 4. Download your model, then put it in GestureGlasses/app/src/main/assets/ (replace the old one)
model.export_model()
files.download('exported_model/gesture_recognizer.task')